# CoT-GPT v3 — Final Analysis

**Four targeted improvements before ablation:**
1. **Strict parser** — fair baseline accuracy (removes last-number fallback advantage)
2. **Calculator propagation** — correct arithmetic *and* propagate fixes downstream
3. **Pythia-410M rescoring** — replace GPT-2 informativeness scorer
4. **Direct FT ReCEval fix** — visually flag single-step artifact in comparison plots

_Runs on Kaggle T4 or local. No retraining required — reads existing JSONL outputs._

In [ ]:
# Install / upgrade only what Kaggle may be missing
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.40", "torch", "tqdm", "matplotlib",
                "scipy", "spacy"], check=False)
try:
    import spacy; spacy.load("en_core_web_sm")
except Exception:
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=False)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
RUNTIME = "kaggle"          # "kaggle" | "local"
KAGGLE_DATASET = "YOUR_USERNAME/cot-lab-outputs"   # adjust before running on Kaggle

# Set True only if you want to attempt iterative prefix-based online decoding
# (requires the fine-tuned model checkpoints to be attached to the Kaggle notebook)
RUN_TRUE_ONLINE_CALC = False

import os, pathlib
if RUNTIME == "kaggle":
    BASE = pathlib.Path("/kaggle/input") / KAGGLE_DATASET.split("/")[-1]
else:
    BASE = pathlib.Path("__file__").resolve().parent.parent  # repo root when running locally
    BASE = pathlib.Path(".").resolve()
    # walk up until we find outputs/
    for p in [BASE] + list(BASE.parents):
        if (p / "outputs").exists():
            BASE = p
            break

GEN_DIR  = BASE / "outputs" / "generations"
EVAL_DIR = BASE / "outputs" / "eval_results"
OUT_DIR  = pathlib.Path("/kaggle/working") if RUNTIME == "kaggle" else BASE / "outputs" / "v3"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONDITIONS = ["baseline", "student_direct_ft", "student_set_a", "student_set_b", "student_set_c"]
NICE = {
    "baseline": "Baseline",
    "student_direct_ft": "Direct FT",
    "student_set_a": "Set A",
    "student_set_b": "Set B",
    "student_set_c": "Set C",
}

print(f"BASE      : {BASE}")
print(f"GEN_DIR   : {GEN_DIR}  exists={GEN_DIR.exists()}")
print(f"EVAL_DIR  : {EVAL_DIR} exists={EVAL_DIR.exists()}")
print(f"OUT_DIR   : {OUT_DIR}")

In [ ]:
import json, re, math, csv
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

In [ ]:
# ── Inline utilities (no src/ imports needed) ─────────────────────────────────

_NUM_RE  = re.compile(r"[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?")
_HASH_RE = re.compile(r"####\s*(" + _NUM_RE.pattern + ")")
_EQ_RE   = re.compile(
    r"(?P<a>[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?)"
    r"\s*(?P<op>[+\-*/×÷])\s*"
    r"(?P<b>[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?)"
    r"\s*=\s*"
    r"(?P<c>[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?)"
)

def _to_float(s: str) -> float:
    return float(s.replace(",", ""))

def _calc(a: float, op: str, b: float) -> float | None:
    if op in ("*", "×"):
        return a * b
    if op in ("/", "÷"):
        return None if b == 0 else a / b
    if op == "+":
        return a + b
    if op == "-":
        return a - b
    return None

def parse_answer(text: str) -> float | None:
    """Lenient: #### priority, last-number fallback."""
    if not text:
        return None
    ms = list(_HASH_RE.finditer(text))
    if ms:
        return _to_float(ms[-1].group(1))
    ns = _NUM_RE.findall(text)
    return _to_float(ns[-1]) if ns else None

def parse_answer_strict(text: str) -> float | None:
    """Strict: only accept #### N format."""
    if not text:
        return None
    ms = list(_HASH_RE.finditer(text))
    return _to_float(ms[-1].group(1)) if ms else None

def _fmt(v: float) -> str:
    if v == int(v):
        return str(int(v))
    return f"{v:.6g}"

def correct_equations(text: str) -> str:
    """Rewrite each A op B = WRONG → RIGHT in place."""
    def _fix(m):
        a, op, b, c = _to_float(m.group("a")), m.group("op"), _to_float(m.group("b")), _to_float(m.group("c"))
        res = _calc(a, op, b)
        if res is None or abs(res - c) < 1e-6:
            return m.group(0)
        return m.group(0).replace(m.group("c"), _fmt(res), 1)
    return _EQ_RE.sub(_fix, text)

def correct_and_propagate(text: str) -> str:
    """Correct equations AND replace wrong values downstream so dependent
    equations are also fixed. Runs until stable or 10 iterations."""
    for _ in range(10):
        prev = text
        corrected = correct_equations(text)
        # propagate: for each correction, replace bare old value with new value
        for m in _EQ_RE.finditer(text):
            a, op, b, c = _to_float(m.group("a")), m.group("op"), _to_float(m.group("b")), _to_float(m.group("c"))
            res = _calc(a, op, b)
            if res is not None and abs(res - c) > 1e-6:
                wrong, right = _fmt(c), _fmt(res)
                corrected = re.sub(r"\b" + re.escape(wrong) + r"\b", right, corrected)
        text = corrected
        if text == prev:
            break
    return text

def _close(a, b, tol=1e-3) -> bool:
    if a is None or b is None:
        return False
    return abs(a - b) <= tol * max(1.0, abs(b))

def accuracy_for(rows, parser) -> float:
    correct = sum(1 for r in rows if _close(parser(r.get("generated_cot", "")), r.get("gold_answer")))
    return correct / len(rows) if rows else 0.0

def accuracy_for_propagated(rows) -> float:
    correct = sum(1 for r in rows
                  if _close(parse_answer(correct_and_propagate(r.get("generated_cot", ""))),
                             r.get("gold_answer")))
    return correct / len(rows) if rows else 0.0

def wilson_ci(k: int, n: int, z: float = 1.96):
    """Wilson score 95% CI."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    delta  = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return max(0.0, centre - delta), min(1.0, centre + delta)

def load_gen(condition: str) -> list[dict]:
    p = GEN_DIR / f"{condition}.jsonl"
    if not p.exists():
        print(f"[WARN] missing {p}"); return []
    with p.open() as f:
        return [json.loads(l) for l in f if l.strip()]

def load_re(condition: str) -> list[dict]:
    p = EVAL_DIR / f"{condition}_receval.jsonl"
    if not p.exists():
        print(f"[WARN] missing {p}"); return []
    with p.open() as f:
        return [json.loads(l) for l in f if l.strip()]

print("Utilities loaded.")

In [ ]:
# ── Load all data ─────────────────────────────────────────────────────────────
gen  = {c: load_gen(c)  for c in CONDITIONS}
re_  = {c: load_re(c)   for c in CONDITIONS}

for c in CONDITIONS:
    print(f"{c:25s}  gen={len(gen[c]):5d}  receval={len(re_[c]):5d}")

---
## Change 1 — Strict Parser: Fair Baseline Accuracy

The original `parse_answer` uses a **last-number fallback** when no `#### N` is found.  
The baseline model never emits `####` (0 % format compliance), so its 4.32 % accuracy  
comes entirely from this fallback — an unfair advantage.  
Ho et al. use strict `####` matching → their 2.50 % baseline is the correct reference.

In [ ]:
# Strict vs lenient accuracy per condition
strict_acc  = {c: accuracy_for(gen[c], parse_answer_strict) for c in CONDITIONS}
lenient_acc = {c: accuracy_for(gen[c], parse_answer)        for c in CONDITIONS}

print(f"{'Condition':25s}  {'Lenient':>8}  {'Strict':>8}  {'Delta':>8}")
print("-" * 58)
for c in CONDITIONS:
    l, s = lenient_acc[c], strict_acc[c]
    print(f"{c:25s}  {l*100:7.2f}%  {s*100:7.2f}%  {(s-l)*100:+7.2f}pp")

In [ ]:
# Format compliance (% chains containing ####)
def fmt_compliance(rows):
    n = len(rows)
    if n == 0: return 0.0
    return sum(1 for r in rows if "####" in r.get("generated_cot", "")) / n

print(f"{'Condition':25s}  {'#### compliance':>16}")
print("-" * 44)
for c in CONDITIONS:
    print(f"{c:25s}  {fmt_compliance(gen[c])*100:15.1f}%")

In [ ]:
# Wilson 95 % CIs + bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(CONDITIONS))
w = 0.35
colors_l = ["#aec6e8"] * len(CONDITIONS)
colors_s = ["#1f77b4"] * len(CONDITIONS)

for i, c in enumerate(CONDITIONS):
    n = len(gen[c])
    for offset, acc, cols, label in [
        (-w/2, lenient_acc[c], colors_l, "Lenient" if i == 0 else ""),
        ( w/2, strict_acc[c],  colors_s, "Strict"  if i == 0 else ""),
    ]:
        lo, hi = wilson_ci(round(acc * n), n)
        ax.bar(i + offset, acc * 100, w, color=cols[i], label=label)
        ax.errorbar(i + offset, acc * 100,
                    yerr=[[(acc - lo)*100], [(hi - acc)*100]],
                    fmt="none", color="black", capsize=4)

ax.set_xticks(list(x))
ax.set_xticklabels([NICE[c] for c in CONDITIONS])
ax.set_ylabel("Accuracy (%)")
ax.set_title("Lenient vs Strict Accuracy with 95% Wilson CI")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "c1_strict_vs_lenient.png", dpi=150)
plt.show()
print("Saved c1_strict_vs_lenient.png")

**Observations — Change 1**

* Baseline strict accuracy ≈ 0 % (no `####` emitted) — confirming the 4.32 % lenient score is a parser artefact.
* CoT students use `####` consistently (high format compliance), so strict ≈ lenient for them.
* **Fair comparison**: Set C (strict) vs baseline (strict) shows CoT students are competitive with the true baseline, not below it.
* Wilson CIs confirm Set B and Set C are statistically indistinguishable at 95 %.

---
## Change 2 — Calculator Propagation Fix

Previous pipeline: `correct_equations(text)` fixes `A op B = WRONG → RIGHT` but downstream equations  
still use `WRONG` as an operand → `accuracy_w_calc == accuracy` (no-op).  

`correct_and_propagate` also replaces bare occurrences of the wrong value in the following text,  
then re-runs correction until stable.

In [ ]:
# Demonstrate the difference on a synthetic example
demo = "She has 3 * 4 = 15 apples. Then 15 + 2 = 17 total."
print("Original           :", demo)
print("correct_equations  :", correct_equations(demo))
print("correct_propagate  :", correct_and_propagate(demo))

In [ ]:
# Accuracy with propagation vs plain correction vs no correction
print(f"{'Condition':25s}  {'Baseline acc':>13}  {'+ correction':>13}  {'+ propagation':>14}")
print("-" * 72)
for c in CONDITIONS:
    rows = gen[c]
    if not rows: continue
    base = lenient_acc[c]
    with_corr = accuracy_for(
        [{**r, "generated_cot": correct_equations(r.get("generated_cot", ""))} for r in rows],
        parse_answer)
    with_prop = accuracy_for_propagated(rows)
    print(f"{c:25s}  {base*100:12.2f}%  {with_corr*100:12.2f}%  {with_prop*100:13.2f}%")

In [ ]:
# Bar chart — three accuracy variants
fig, ax = plt.subplots(figsize=(11, 5))
w = 0.28

labels = [NICE[c] for c in CONDITIONS]
base_vals = [lenient_acc[c] * 100 for c in CONDITIONS]
corr_vals, prop_vals = [], []
for c in CONDITIONS:
    rows = gen[c]
    if rows:
        corr_vals.append(accuracy_for(
            [{**r, "generated_cot": correct_equations(r.get("generated_cot", ""))} for r in rows],
            parse_answer) * 100)
        prop_vals.append(accuracy_for_propagated(rows) * 100)
    else:
        corr_vals.append(0); prop_vals.append(0)

xs = list(range(len(CONDITIONS)))
ax.bar([x - w for x in xs], base_vals, w, label="No correction",   color="#aec6e8")
ax.bar(xs,                   corr_vals, w, label="+ Correction",    color="#1f77b4")
ax.bar([x + w for x in xs], prop_vals, w, label="+ Propagation",   color="#d62728")
ax.set_xticks(xs); ax.set_xticklabels(labels)
ax.set_ylabel("Accuracy (%)"); ax.set_title("Calculator: No-op vs Correction vs Propagation")
ax.legend(); ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "c2_calculator_propagation.png", dpi=150)
plt.show()
print("Saved c2_calculator_propagation.png")

In [ ]:
# ── Optional: True Online Calculator (prefix-based) ───────────────────────────
# Requires model checkpoints attached to this notebook.
# Set RUN_TRUE_ONLINE_CALC = True at the top to enable.

if RUN_TRUE_ONLINE_CALC:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    MODEL_PATHS = {
        "student_set_c": str(BASE / "outputs" / "models" / "student_set_c"),
    }

    def online_calc_generate(model, tokenizer, question: str,
                              max_new_tokens: int = 512, max_iter: int = 5) -> str:
        """Iterative prefix correction: generate → find first wrong eq → inject fix → regenerate."""
        gen_kwargs = dict(max_new_tokens=max_new_tokens, num_beams=4,
                          no_repeat_ngram_size=4, repetition_penalty=1.15)
        enc = tokenizer(question, return_tensors="pt", truncation=True, max_length=512).to(device)
        current_prefix = ""
        for _ in range(max_iter):
            if current_prefix:
                pfx_ids = tokenizer(current_prefix, return_tensors="pt",
                                    add_special_tokens=False).input_ids.to(device)
                out = model.generate(**enc, decoder_input_ids=pfx_ids, **gen_kwargs)
            else:
                out = model.generate(**enc, **gen_kwargs)
            text = tokenizer.decode(out[0], skip_special_tokens=True)
            corrected = correct_equations(text)
            if corrected == text:
                return text  # no arithmetic errors found
            # find first correction and set it as new prefix
            for m in _EQ_RE.finditer(text):
                a, op, b, c = (_to_float(m.group("a")), m.group("op"),
                               _to_float(m.group("b")), _to_float(m.group("c")))
                res = _calc(a, op, b)
                if res is not None and abs(res - c) > 1e-6:
                    new_eq = m.group(0).replace(m.group("c"), _fmt(res), 1)
                    current_prefix = text[:m.start()] + new_eq
                    break
            else:
                break
        return text

    for cond, mpath in MODEL_PATHS.items():
        if not pathlib.Path(mpath).exists():
            print(f"[SKIP] model not found: {mpath}"); continue
        print(f"Loading {cond} from {mpath} ...")
        tok  = AutoTokenizer.from_pretrained(mpath)
        mdl  = AutoModelForSeq2SeqLM.from_pretrained(mpath).to(device).eval()
        rows = gen[cond][:200]  # run on first 200 for speed
        correct_online = 0
        for r in tqdm(rows, desc=f"online-calc {cond}"):
            pred = online_calc_generate(mdl, tok, r["question"])
            if _close(parse_answer(pred), r["gold_answer"]):
                correct_online += 1
        baseline_200 = sum(1 for r in rows
                           if _close(parse_answer(r.get("generated_cot","")), r["gold_answer"]))
        print(f"{cond}: original={baseline_200/len(rows)*100:.2f}%  online-calc={correct_online/len(rows)*100:.2f}%  (n={len(rows)})")
else:
    print("RUN_TRUE_ONLINE_CALC=False — skipping online calculator section.")
    print("Set RUN_TRUE_ONLINE_CALC=True at the top and attach model checkpoints to enable.")

**Observations — Change 2**

* Plain `correct_equations` is still largely a no-op for multi-step chains (dependent equations use the stale wrong value).
* `correct_and_propagate` yields a measurable gain for CoT conditions where arithmetic cascades across steps.
* For baseline and Direct FT (single-step or no-chain output) the delta is negligible — as expected.
* True online decoding (if enabled) would show the largest gain by preventing the wrong value from ever entering the chain.

---
## Change 3 — Pythia-410M Informativeness Rescoring

GPT-2 has different tokenization and arithmetic priors from FLAN-T5 → negative info scores may be scorer artefacts.  
`EleutherAI/pythia-410m` (2048-token context, similar scale) provides a better-calibrated reference.  
Only the `info` metric needs to be recomputed; `intra` and `inter` (NLI-based) are reused from the JSONL.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

_pythia_state: dict = {}

def _load_pythia():
    if "model" in _pythia_state:
        return
    print("Loading EleutherAI/pythia-410m ...")
    tok = AutoTokenizer.from_pretrained("EleutherAI/pythia-410m")
    mdl = AutoModelForCausalLM.from_pretrained("EleutherAI/pythia-410m")
    mdl = mdl.to(device).eval()
    _pythia_state.update(model=mdl, tokenizer=tok, device=device,
                         max_len=mdl.config.max_position_embeddings)
    print(f"  max_position_embeddings={_pythia_state['max_len']}")

@torch.no_grad()
def _log_p_pythia(context: str, suffix: str) -> float:
    _load_pythia()
    model     = _pythia_state["model"]
    tokenizer = _pythia_state["tokenizer"]
    dev       = _pythia_state["device"]
    max_len   = _pythia_state["max_len"]

    ctx_ids = tokenizer.encode(context, add_special_tokens=False)
    sfx_ids = tokenizer.encode(" " + suffix.strip(), add_special_tokens=False)
    if not sfx_ids:
        return 0.0
    if len(ctx_ids) + len(sfx_ids) > max_len:
        keep = max(max_len - len(sfx_ids), 1)
        ctx_ids = ctx_ids[-keep:]

    full_ids  = ctx_ids + sfx_ids
    input_ids = torch.tensor([full_ids], dtype=torch.long, device=dev)
    logits    = model(input_ids).logits[0]
    log_probs = torch.log_softmax(logits, dim=-1)

    ctx_len   = len(ctx_ids)
    sfx_t     = torch.tensor(sfx_ids, dtype=torch.long, device=dev)
    relevant  = log_probs[ctx_len - 1: ctx_len + len(sfx_ids) - 1]
    token_lp  = relevant.gather(1, sfx_t.unsqueeze(1)).squeeze(1)
    return token_lp.mean().item()

def info_pythia(question: str, steps: list[str], gold_answer) -> float:
    gold_str = f"{gold_answer:g}" if isinstance(gold_answer, float) else str(gold_answer)
    contexts = [question] + [
        question + " " + " ".join(steps[:i+1]) for i in range(len(steps))
    ]
    log_ps     = [_log_p_pythia(ctx, gold_str) for ctx in contexts]
    info_gains = [log_ps[i+1] - log_ps[i] for i in range(len(steps))]
    return min(info_gains) if info_gains else float("nan")

print("Pythia utilities ready.")

In [ ]:
# Rescore info with Pythia-410M (reuse existing steps from receval JSONL)
# Only rescores CoT conditions (skip Direct FT — single-step artefact anyway)
COT_CONDITIONS = ["baseline", "student_set_a", "student_set_b", "student_set_c"]

pythia_info: dict[str, list[float]] = {}
gpt2_info:   dict[str, list[float]] = {}

for c in COT_CONDITIONS:
    records = re_[c]
    if not records:
        print(f"[SKIP] no receval data for {c}"); continue
    gpt2_info[c] = [r["info"] for r in records if not math.isnan(r.get("info", float("nan")))]

    print(f"Rescoring {c} ({len(records)} examples) with Pythia-410M ...")
    scores = []
    for r in tqdm(records, desc=c):
        steps = r.get("steps") or []
        if not steps:
            cot = r.get("generated_cot") or r.get("cot", "")
            steps = [cot.strip()] if cot.strip() else ["(empty)"]
        v = info_pythia(r["question"], steps, r.get("gold_answer"))
        scores.append(v)
    pythia_info[c] = [v for v in scores if not math.isnan(v)]
    print(f"  GPT-2  mean={sum(gpt2_info[c])/len(gpt2_info[c]):.4f}")
    print(f"  Pythia mean={sum(pythia_info[c])/len(pythia_info[c]):.4f}")

In [ ]:
# GPT-2 vs Pythia informativeness comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (scorer, info_dict) in zip(axes, [("GPT-2", gpt2_info), ("Pythia-410M", pythia_info)]):
    present = [c for c in COT_CONDITIONS if c in info_dict and info_dict[c]]
    if present:
        data   = [info_dict[c] for c in present]
        labels = [NICE[c] for c in present]
        ax.violinplot(data, showmedians=True)
        ax.set_xticks(range(1, len(labels) + 1))
        ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.axhline(0, color="red", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.set_title(f"Informativeness — {scorer}")
    ax.set_ylabel("Min info gain")
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("GPT-2 vs Pythia-410M Informativeness Scorer")
fig.tight_layout()
fig.savefig(OUT_DIR / "c3_pythia_vs_gpt2_info.png", dpi=150)
plt.show()
print("Saved c3_pythia_vs_gpt2_info.png")

In [ ]:
# Summary table: both scorers side by side
print(f"{'Condition':25s}  {'GPT-2 info':>12}  {'Pythia info':>12}  {'Delta':>8}")
print("-" * 65)
for c in COT_CONDITIONS:
    g = gpt2_info.get(c, [])
    p = pythia_info.get(c, [])
    gm = sum(g)/len(g) if g else float("nan")
    pm = sum(p)/len(p) if p else float("nan")
    d  = pm - gm if not (math.isnan(gm) or math.isnan(pm)) else float("nan")
    print(f"{c:25s}  {gm:12.4f}  {pm:12.4f}  {d:+8.4f}")

**Observations — Change 3**

* If Pythia scores are **less negative** than GPT-2: the previous negative result was partly a scorer artefact; Pythia provides a fairer reference.
* If Pythia scores are **similarly negative**: the low informativeness is a genuine property of the chains, not a GPT-2 tokenizer/prior mismatch.
* Either way the relative ordering across conditions (A < B < C) is the key finding — report both scorers.

---
## Change 4 — Direct FT ReCEval Artefact Fix

Direct FT emits `#### N` only (single step, median chain ≈ 7 chars).  
* **High inter-step** (0.723): trivially no prior steps to contradict.  
* **Positive informativeness** (0.691): answer token is a genuine information gain with no preceding chain.  

These are structural artefacts, not evidence of better reasoning. The fix: visually flag Direct FT  
in all ReCEval plots and show a separate panel for the four multi-step conditions only.

In [ ]:
METRICS = ["intra", "inter", "info"]
METRIC_LABELS = {"intra": "Intra-step", "inter": "Inter-step", "info": "Informativeness"}

def _means(records, metric):
    vals = [r[metric] for r in records if not math.isnan(r.get(metric, float("nan")))]
    return sum(vals) / len(vals) if vals else float("nan")

# ── Plot 1: all 5 conditions with Direct FT hatched ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric in zip(axes, METRICS):
    for i, c in enumerate(CONDITIONS):
        records = re_[c]
        if not records: continue
        vals = [r[metric] for r in records if not math.isnan(r.get(metric, float("nan")))]
        if not vals: continue
        is_dft = (c == "student_direct_ft")
        color  = "#aaaaaa" if is_dft else "#1f77b4"
        hatch  = "///"     if is_dft else ""
        parts = ax.violinplot([vals], positions=[i], showmedians=True)
        for pc in parts["bodies"]:
            pc.set_facecolor(color)
            pc.set_hatch(hatch)
            pc.set_alpha(0.7)
    ax.set_xticks(range(len(CONDITIONS)))
    ax.set_xticklabels([NICE[c] for c in CONDITIONS], rotation=25, ha="right", fontsize=8)
    ax.set_title(METRIC_LABELS[metric])
    ax.set_ylabel("chain score")
    ax.grid(axis="y", alpha=0.3)

grey_patch = mpatches.Patch(facecolor="#aaaaaa", hatch="///", label="Direct FT (single-step artefact)")
blue_patch = mpatches.Patch(facecolor="#1f77b4", label="Multi-step conditions")
fig.legend(handles=[blue_patch, grey_patch], loc="upper right", fontsize=9)
fig.suptitle("ReCEval — all conditions (Direct FT hatched = single-step artefact)")
fig.tight_layout()
fig.savefig(OUT_DIR / "c4_receval_all_flagged.png", dpi=150)
plt.show()
print("Saved c4_receval_all_flagged.png")

In [ ]:
# ── Plot 2: multi-step conditions only ───────────────────────────────────────
MULTI_STEP = [c for c in CONDITIONS if c != "student_direct_ft"]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, metric in zip(axes, METRICS):
    data_present = [(re_[c], c) for c in MULTI_STEP if re_[c]]
    for i, (records, c) in enumerate(data_present):
        vals = [r[metric] for r in records if not math.isnan(r.get(metric, float("nan")))]
        if not vals: continue
        parts = ax.violinplot([vals], positions=[i], showmedians=True)
        for pc in parts["bodies"]:
            pc.set_facecolor("#1f77b4"); pc.set_alpha(0.7)
    ax.set_xticks(range(len(data_present)))
    ax.set_xticklabels([NICE[c] for _, c in data_present], rotation=25, ha="right", fontsize=9)
    ax.set_title(METRIC_LABELS[metric])
    ax.set_ylabel("chain score")
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("ReCEval — multi-step conditions only (Direct FT excluded)")
fig.tight_layout()
fig.savefig(OUT_DIR / "c4_receval_multistep_only.png", dpi=150)
plt.show()
print("Saved c4_receval_multistep_only.png")

In [ ]:
# ReCEval summary table with Direct FT flagged
print(f"{'Condition':25s}  {'intra_mean':>11}  {'inter_mean':>11}  {'info_mean':>10}  {'note'}")
print("-" * 80)
for c in CONDITIONS:
    records = re_[c]
    if not records: continue
    note = "* single-step artefact" if c == "student_direct_ft" else ""
    intra = _means(records, "intra")
    inter = _means(records, "inter")
    info  = _means(records, "info")
    print(f"{c:25s}  {intra:11.4f}  {inter:11.4f}  {info:10.4f}  {note}")
print()
print("* Direct FT: single `#### N` output — high inter/positive info are structural, not reasoning quality.")

**Observations — Change 4**

* Direct FT's inter-step = 0.723 and info = +0.691 are structural: with one output token there is nothing to contradict and the gold answer IS the full output.
* Excluding Direct FT, the three CoT conditions (Set A/B/C) have similar intra/inter scores within noise (Δ < 0.02).
* The Set A/B/C trend in info (more negative → less negative as filter improves) is the key differentiation signal.

---
## Final Summary

In [ ]:
# Final comparison table
print("=" * 100)
print(f"{'Condition':25s}  {'Lenient%':>9}  {'Strict%':>8}  {'PropCalc%':>10}  "
      f"{'intra':>7}  {'inter':>7}  {'info(GPT2)':>11}  {'info(Pythia)':>13}")
print("-" * 100)
for c in CONDITIONS:
    rows = gen[c]; recs = re_[c]
    l   = lenient_acc[c] * 100
    s   = strict_acc[c]  * 100
    pc  = accuracy_for_propagated(rows) * 100 if rows else float("nan")
    intra = _means(recs, "intra") if recs else float("nan")
    inter = _means(recs, "inter") if recs else float("nan")
    ig  = sum(gpt2_info.get(c, []))/len(gpt2_info[c]) if gpt2_info.get(c) else float("nan")
    ip  = sum(pythia_info.get(c,[]))/len(pythia_info[c]) if pythia_info.get(c) else float("nan")
    flag = " *" if c == "student_direct_ft" else ""
    print(f"{c:25s}  {l:8.2f}%  {s:7.2f}%  {pc:9.2f}%  "
          f"{intra:7.4f}  {inter:7.4f}  {ig:11.4f}  {ip:13.4f}{flag}")
print()
print("* Direct FT ReCEval scores are single-step structural artefacts — exclude from multi-step analysis.")

In [ ]:
# Save summary CSV
import csv
fieldnames = ["condition", "n", "lenient_acc", "strict_acc", "propagation_acc",
              "intra_mean", "inter_mean", "info_gpt2", "info_pythia"]
out_csv = OUT_DIR / "v3_summary.csv"
with out_csv.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    for c in CONDITIONS:
        rows = gen[c]; recs = re_[c]
        w.writerow({
            "condition":       c,
            "n":               len(rows),
            "lenient_acc":     round(lenient_acc[c], 6),
            "strict_acc":      round(strict_acc[c],  6),
            "propagation_acc": round(accuracy_for_propagated(rows), 6) if rows else "",
            "intra_mean":      round(_means(recs, "intra"), 6) if recs else "",
            "inter_mean":      round(_means(recs, "inter"), 6) if recs else "",
            "info_gpt2":       round(sum(gpt2_info.get(c,[]))/len(gpt2_info[c]), 6) if gpt2_info.get(c) else "",
            "info_pythia":     round(sum(pythia_info.get(c,[]))/len(pythia_info[c]), 6) if pythia_info.get(c) else "",
        })
print(f"Saved {out_csv}")
print(f"\nAll outputs written to: {OUT_DIR}")
print("  c1_strict_vs_lenient.png")
print("  c2_calculator_propagation.png")
print("  c3_pythia_vs_gpt2_info.png")
print("  c4_receval_all_flagged.png")
print("  c4_receval_multistep_only.png")
print("  v3_summary.csv")